# 15.2 `unittest` — the Standard Library

**Prerequisites:** 15.1 Why Test and the assert Statement, 05 OOPs, 06 Exception Handling  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- `TestCase` — the class-based test, and why it looks the way it does
- The `assert*` method family, and which ones actually earn their place
- 🔴 **failure vs error** — a distinction `pytest` deliberately drops
- `setUp` / `tearDown` / `setUpClass` and the exact order they run in
- `subTest` — many cases in one test, all of them reported
- `assertRaises`, `assertRaisesRegex`, `assertLogs`, `assertWarns`
- Skips and expected failures
- 🔴 The aliases removed in Python 3.12 — `assertEquals` is gone
- `python -m unittest discover`, and when to pick `unittest` over `pytest`

---

## Why start here

`unittest` ships with Python. No install, no dependency, available on every machine that has
an interpreter — including the locked-down build server where you cannot `pip install`
anything.

It is a port of **JUnit**, which is why it looks un-Pythonic: classes, camelCase methods,
`setUp` rather than `set_up`. That heritage is worth knowing, because it explains everything
odd about the API.

| | `unittest` | `pytest` (**15.3**) |
|---|---|---|
| Ships with Python | ✅ | ❌ `pip install pytest` |
| Tests are | methods on a `TestCase` subclass | plain functions |
| Assertions | `self.assertEqual(a, b)` | `assert a == b` |
| Failure output | "a != b" | full diff of both values |
| Setup | `setUp` method | fixtures, injected per test |
| Parametrisation | `subTest` (clumsy) | `@parametrize` (excellent) |

**You need both.** `pytest` runs `unittest` tests unchanged, so real projects often contain
both — and you will read `unittest` code for as long as you read Python.

## The code under test

One small module, used for the whole notebook: the state machine of a background job.

```
   queued ──start──> running ──finish──> done
      │                  │
      └──cancel──────────┴──fail──> failed
```

In [ ]:
VALID_TRANSITIONS = {
    "queued":  {"running", "failed"},
    "running": {"done", "failed"},
    "done":    set(),
    "failed":  {"queued"},          # a failed job may be requeued
}


class Job:
    """A background job with a state machine and a retry budget."""

    def __init__(self, job_id, max_retries=3):
        self.job_id = job_id
        self.state = "queued"
        self.max_retries = max_retries
        self.attempts = 0

    def advance(self, new_state):
        if new_state not in VALID_TRANSITIONS:
            raise ValueError(f"unknown state {new_state!r}")
        if new_state not in VALID_TRANSITIONS[self.state]:
            raise ValueError(f"cannot go from {self.state!r} to {new_state!r}")
        if new_state == "running":
            self.attempts += 1
        self.state = new_state
        return self

    @property
    def can_retry(self):
        return self.state == "failed" and self.attempts < self.max_retries

    def __repr__(self):
        return f"Job({self.job_id!r}, state={self.state!r}, attempts={self.attempts})"


print(Job("build-42").advance("running").advance("done"))

## Your first `TestCase`

```
class JobStateTests(unittest.TestCase):     subclass TestCase — this is what makes it a test
    def test_new_job_starts_queued(self):   method name must start with `test`
        job = Job("build-1")                    ARRANGE
        self.assertEqual(job.state, "queued")   ACT + ASSERT
    ──┬─ ─────┬───── ──────┬──── ────┬───
      │       │            │         └─ expected
      │       │            └─ actual  (order matters for the message, not the result)
      │       └─ the assertion method
      └─ every assertion is a method on self
```

### 🔴 Running it inside a notebook

The usual entry point is `unittest.main()` — but in a notebook that **calls `sys.exit()`**
and would kill your kernel. Two ways round it:

```python
unittest.main(argv=["ignored"], exit=False)     # works, but collects every TestCase defined
```

or — clearer, and what this notebook uses — load one class explicitly and run it:

In [ ]:
import io
import unittest


def run_case(test_class, verbosity=2):
    """Run one TestCase and print the report. Returns the TestResult."""
    suite = unittest.TestLoader().loadTestsFromTestCase(test_class)
    stream = io.StringIO()
    result = unittest.TextTestRunner(stream=stream, verbosity=verbosity).run(suite)
    print(stream.getvalue().rstrip())
    print(f"\n  --> ran {result.testsRun}"
          f" | failures {len(result.failures)}"
          f" | errors {len(result.errors)}"
          f" | skipped {len(result.skipped)}"
          f" | expected failures {len(result.expectedFailures)}")
    return result


class JobStateTests(unittest.TestCase):

    def test_new_job_starts_queued(self):
        job = Job("build-1")

        self.assertEqual(job.state, "queued")

    def test_starting_a_job_counts_an_attempt(self):
        job = Job("build-1")

        job.advance("running")

        self.assertEqual(job.attempts, 1)

    def test_failed_job_can_be_requeued(self):
        job = Job("build-1").advance("running").advance("failed")

        job.advance("queued")

        self.assertEqual(job.state, "queued")


run_case(JobStateTests)

## The `assert*` family

There are about 35 of them. These are the ones worth memorising:

| Method | Passes when | Use it for |
|---|---|---|
| `assertEqual(a, b)` | `a == b` | the default; the workhorse |
| `assertNotEqual(a, b)` | `a != b` | |
| `assertTrue(x)` / `assertFalse(x)` | `bool(x)` | 🔴 last resort — the message tells you nothing |
| `assertIs(a, b)` / `assertIsNot` | `a is b` | sentinels, `None`, enum members |
| `assertIsNone(x)` / `assertIsNotNone` | `x is None` | |
| `assertIn(a, b)` / `assertNotIn` | `a in b` | substrings, membership |
| `assertIsInstance(a, cls)` | `isinstance(a, cls)` | |
| `assertAlmostEqual(a, b)` | rounds to 7 places | 🔴 **floats** — never `assertEqual` |
| `assertCountEqual(a, b)` | same elements, any order | comparing unordered results |
| `assertRaises(exc)` | the block raises `exc` | as a context manager |
| `assertRaisesRegex(exc, rx)` | …and the message matches | |
| `assertLogs(logger, level)` | something was logged | |
| `assertWarns(cls)` | a warning was issued | |
| `assertMultiLineEqual(a, b)` | strings equal, **with a diff** | used automatically by `assertEqual` on `str` |

🔴 **`assertTrue` is the trap.** `self.assertTrue(job.state == "done")` fails with
*"False is not true"* — useless. `self.assertEqual(job.state, "done")` fails with
*"'running' != 'done'"* — which tells you what went wrong. Always prefer the specific method.

In [ ]:
class AssertionTourTests(unittest.TestCase):

    def test_equality_and_membership(self):
        job = Job("etl-9")
        self.assertEqual(job.job_id, "etl-9")
        self.assertNotEqual(job.state, "done")
        self.assertIn("etl", job.job_id)
        self.assertIsInstance(job.attempts, int)
        self.assertIsNone(getattr(job, "finished_at", None))

    def test_identity_vs_equality(self):
        a, b = ["queued"], ["queued"]
        self.assertEqual(a, b)          # same contents
        self.assertIsNot(a, b)          # different objects  (see 2.7)

    def test_floats_need_almost_equal(self):
        total = 0.1 + 0.2
        self.assertNotEqual(total, 0.3)             # 🔴 exact comparison FAILS
        self.assertAlmostEqual(total, 0.3)          # this is the right way
        self.assertAlmostEqual(total, 0.3, places=15)
        print(f"      0.1 + 0.2 = {total!r}")

    def test_order_insensitive_comparison(self):
        finished = ["etl-9", "build-1", "etl-9"]
        expected = ["etl-9", "etl-9", "build-1"]
        self.assertNotEqual(finished, expected)     # lists compare positionally
        self.assertCountEqual(finished, expected)   # same multiset -> passes


run_case(AssertionTourTests)

### 🔴 Why `assertEqual` beats `assertTrue` — shown, not asserted

Two tests for the same broken behaviour. Read the two failure messages.

In [ ]:
class MessageQualityTests(unittest.TestCase):

    def test_with_assert_true(self):
        job = Job("build-7").advance("running")
        self.assertTrue(job.state == "done")

    def test_with_assert_equal(self):
        job = Job("build-7").advance("running")
        self.assertEqual(job.state, "done")


run_case(MessageQualityTests, verbosity=0)
print("\n  Same bug. The first message cannot be debugged; the second names the value.")

## 🔴 Failure vs error

`unittest` reports two different things, and the difference is diagnostic:

| | Means | Cause |
|---|---|---|
| **FAIL** | the code ran, and gave the wrong answer | an `assert*` method did not hold |
| **ERROR** | the code blew up before we could check | any *other* exception — including in `setUp` |

An ERROR usually means the test itself is broken, or the code crashed somewhere you were not
looking. A FAIL means the code is wrong in the specific way you predicted.

You saw the same split in **15.1**'s hand-rolled runner. 🔴 `pytest` **discards** this
distinction — everything is a failure — which is a real (if small) loss.

In [ ]:
class FailureVersusErrorTests(unittest.TestCase):

    def test_this_is_a_failure(self):
        job = Job("build-3")
        self.assertEqual(job.state, "running")      # wrong answer -> FAIL

    def test_this_is_an_error(self):
        job = Job("build-3")
        job.advance("done")                         # illegal transition -> raises -> ERROR
        self.assertEqual(job.state, "done")


def last_exception_line(report):
    """The final `SomeError: message` line of a traceback report."""
    return next(line for line in reversed(report.strip().splitlines())
                if line and not line.startswith((" ", "+", "-", "?")))


result = run_case(FailureVersusErrorTests, verbosity=0)
print("\n  reported as FAILURE:", last_exception_line(result.failures[0][1]))
print("  reported as ERROR  :", last_exception_line(result.errors[0][1]))

## The lifecycle: `setUp`, `tearDown`, and their class-level cousins

Most tests need the same starting conditions. `setUp` runs **before every test method**;
`tearDown` runs **after every one**, even if the test failed.

```
setUpClass()                once, before any test in the class
    │
    ├── setUp() ─> test_a() ─> tearDown()
    ├── setUp() ─> test_b() ─> tearDown()
    └── setUp() ─> test_c() ─> tearDown()
    │
tearDownClass()             once, after the last test
```

🔴 **Tests run in alphabetical order by method name**, not in the order you wrote them. Never
rely on the order — that was the bug demonstrated at the end of **15.1**.

The next cell prints the sequence as it happens, so you can read the order rather than trust
the diagram.

In [ ]:
TRACE = []


class LifecycleTests(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        TRACE.append("setUpClass")
        cls.shared_registry = {"jobs": 0}       # expensive, built once

    @classmethod
    def tearDownClass(cls):
        TRACE.append("tearDownClass")

    def setUp(self):
        TRACE.append("  setUp")
        self.job = Job("build-1")               # cheap, fresh for every test

    def tearDown(self):
        TRACE.append("  tearDown")

    def test_b_second_alphabetically(self):
        TRACE.append("    test_b")
        self.assertEqual(self.job.state, "queued")

    def test_a_first_alphabetically(self):
        TRACE.append("    test_a")
        self.job.advance("running")
        self.assertEqual(self.job.attempts, 1)

    def test_c_isolation_check(self):
        TRACE.append("    test_c")
        self.assertEqual(self.job.attempts, 0)   # test_a's advance did NOT leak


run_case(LifecycleTests, verbosity=0)
print("\n  execution order:")
for step in TRACE:
    print("   ", step)

Two things to take from that trace:

1. **`test_a` ran before `test_b`** even though `test_b` is defined first — alphabetical order.
2. **`test_c` saw `attempts == 0`** despite `test_a` having advanced a job. Each test got its
   own `Job` from `setUp`. That is isolation, and it is why `setUp` exists.

🔴 **`setUpClass` state is shared and mutable.** `cls.shared_registry` above is created once;
if one test mutates it, later tests see the change. Use class-level setup only for things that
are genuinely read-only or genuinely expensive. **15.4** shows the same trap with pytest's
session-scoped fixtures.

> `addCleanup(fn, *args)` is often better than `tearDown`: it registers cleanup **at the moment
> you create the thing**, so it runs even if `setUp` fails halfway through.

The next cell registers three cleanups per test and logs the order, so you can check two claims
against the output rather than take them on trust: cleanups run **last-registered-first**
(LIFO), and they run **even when the test fails**.

In [ ]:
import tempfile
import shutil
from pathlib import Path


CLEANUP_LOG = []


class CleanupTests(unittest.TestCase):

    def setUp(self):
        self.workdir = Path(tempfile.mkdtemp(prefix="py152_"))
        self.addCleanup(CLEANUP_LOG.append, "  1st registered")
        self.addCleanup(shutil.rmtree, self.workdir, ignore_errors=True)
        self.addCleanup(CLEANUP_LOG.append, "  3rd registered")
        self.spool = self.workdir / "spool"
        self.spool.mkdir()

    def test_a_job_writes_its_payload(self):
        CLEANUP_LOG.append("test_a body")
        (self.spool / "build-1.json").write_text('{"state": "queued"}', encoding="utf-8")
        self.assertEqual(len(list(self.spool.iterdir())), 1)

    def test_b_spool_starts_empty_every_time(self):
        CLEANUP_LOG.append("test_b body")
        self.assertEqual(list(self.spool.iterdir()), [])

    def test_c_cleanup_runs_even_when_the_test_fails(self):
        CLEANUP_LOG.append("test_c body (about to fail)")
        self.fail("deliberate failure, to prove cleanup still runs")


run_case(CleanupTests, verbosity=0)
print("\n  what actually happened, in order:")
for step in CLEANUP_LOG:
    print("   ", step)

## `subTest` — many cases, one test, all reported

Testing a function against a table of inputs with a plain `for` loop has a fatal flaw:
**the first failure ends the test** and hides every later case.

`with self.subTest(...)` makes each iteration independently reportable. The keyword arguments
you pass are printed with the failure, so you know *which* case broke.

In [ ]:
def parse_retry_after(header, default=0):
    """Seconds from a Retry-After header. 🔴 Deliberately naive - it only
    understands bare integers, and silently returns `default` otherwise."""
    if header.isdigit():
        return int(header)
    return default


CASES = [("1", 1), ("30", 30), ("2.5", 2), (" 5 ", 5), ("abc", 0)]


class PlainLoopTests(unittest.TestCase):
    def test_all_cases_in_one_loop(self):
        for raw, expected in CASES:
            self.assertEqual(parse_retry_after(raw), expected)


class SubTestTests(unittest.TestCase):
    def test_all_cases_with_subtest(self):
        for raw, expected in CASES:
            with self.subTest(raw=raw, expected=expected):
                self.assertEqual(parse_retry_after(raw), expected)


print("### plain loop - stops at the first bad case")
run_case(PlainLoopTests, verbosity=0)

print("\n### subTest - every bad case reported")
run_case(SubTestTests, verbosity=0)

The plain loop reported **one** failure and stopped. `subTest` reported **both**
`'2.5'` and `' 5 '` — and named them.

That difference is the whole argument. When a table-driven test fails you want the complete
list of broken cases, not the first one alphabetically.

🔴 **`subTest` still counts as one test.** `testsRun` is 1 either way; the sub-cases are
reported but not counted. `pytest`'s `@parametrize` (**15.3**) generates *real* separate tests
instead, which is why it is the better tool when you have it.

## Asserting on exceptions, logs and warnings

These three come up constantly and are easy to get wrong.

```python
with self.assertRaises(ValueError):        # ✅ the block must raise ValueError
    job.advance("done")

self.assertRaises(ValueError, job.advance, "done")   # older callable form
```

🔴 **Do not put more than the one failing call inside the block.** If two lines in the block
can raise `ValueError`, the test passes for the wrong reason.

In [ ]:
import logging
import warnings


def requeue(job, logger):
    """Requeue a failed job, warning if it is out of retries."""
    if job.state != "failed":
        raise ValueError(f"only failed jobs can be requeued, not {job.state!r}")
    if not job.can_retry:
        logger.warning("job %s exhausted its retry budget", job.job_id)
        warnings.warn("retry budget exhausted", RuntimeWarning, stacklevel=2)
        return False
    job.advance("queued")
    return True


class ExceptionsLogsWarningsTests(unittest.TestCase):

    def test_requeueing_a_running_job_is_rejected(self):
        job = Job("build-1").advance("running")

        with self.assertRaises(ValueError):
            requeue(job, logging.getLogger("worker"))

    def test_the_error_message_names_the_state(self):
        job = Job("build-1").advance("running")

        with self.assertRaisesRegex(ValueError, r"only failed jobs.*'running'"):
            requeue(job, logging.getLogger("worker"))

    def test_exhausted_budget_is_logged(self):
        job = Job("build-1", max_retries=1).advance("running").advance("failed")
        logger = logging.getLogger("worker")

        with self.assertLogs(logger, level="WARNING") as captured:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                requeue(job, logger)

        self.assertEqual(len(captured.output), 1)
        self.assertIn("exhausted its retry budget", captured.output[0])
        print("      captured log:", captured.output[0])

    def test_exhausted_budget_warns(self):
        job = Job("build-1", max_retries=1).advance("running").advance("failed")

        with self.assertLogs("worker", level="WARNING"):
            with self.assertWarns(RuntimeWarning) as caught:
                requeue(job, logging.getLogger("worker"))

        self.assertEqual(str(caught.warning), "retry budget exhausted")

    def test_exception_object_is_available_for_inspection(self):
        job = Job("build-1")

        with self.assertRaises(ValueError) as caught:
            job.advance("nonsense")

        self.assertIn("unknown state", str(caught.exception))
        print("      caught:", caught.exception)


run_case(ExceptionsLogsWarningsTests, verbosity=0)

> **Note on `assertLogs`.** It requires that *something* was logged — an empty block is a
> failure, not a pass. To assert the opposite (nothing was logged) use `assertNoLogs`,
> added in **Python 3.10**.

## Skipping and expected failures

Not every test applies everywhere: some need a server, a platform, an optional package.

| Decorator / call | Meaning |
|---|---|
| `@unittest.skip("reason")` | never run this |
| `@unittest.skipIf(cond, "reason")` | skip when `cond` |
| `@unittest.skipUnless(cond, "reason")` | run **only** when `cond` |
| `self.skipTest("reason")` | decide at runtime, inside the test |
| `@unittest.expectedFailure` | 🔴 known bug: **passing is now the failure** |

`expectedFailure` is the interesting one. It documents a bug you have not fixed yet, and the
moment someone fixes it the suite tells you — `unexpected success` — so you can delete the
marker. That is much better than commenting the test out.

In [ ]:
import sys


class SkipAndExpectedFailureTests(unittest.TestCase):

    @unittest.skip("no message broker in this environment")
    def test_needs_a_broker(self):
        raise AssertionError("never runs")

    @unittest.skipUnless(sys.platform.startswith("win"), "windows-only path handling")
    def test_windows_spool_path(self):
        self.assertTrue(True)

    @unittest.skipIf(sys.version_info < (3, 12), "requires 3.12+")
    def test_modern_python_only(self):
        self.assertTrue(True)

    def test_decides_at_runtime(self):
        if not hasattr(Job, "priority"):
            self.skipTest("Job has no priority field yet")
        raise AssertionError("never runs")

    @unittest.expectedFailure
    def test_known_bug_float_retry_after(self):
        # parse_retry_after mishandles "2.5" - documented, not yet fixed.
        self.assertEqual(parse_retry_after("2.5"), 2)

    @unittest.expectedFailure
    def test_bug_that_someone_quietly_fixed(self):
        self.assertEqual(parse_retry_after("30"), 30)      # this actually passes!


result = run_case(SkipAndExpectedFailureTests)
print("\n  unexpected successes:", len(result.unexpectedSuccesses),
      "->", [t._testMethodName for t in result.unexpectedSuccesses])

Note the last line: `test_bug_that_someone_quietly_fixed` is marked
`@expectedFailure` but **passed**, so `unittest` reports an **unexpected success**. That is
your cue to delete the decorator.

🔴 A skipped test is a test that is not protecting you. Skips are for genuinely conditional
things (platform, optional dependency); if you are skipping to silence a failure, you have
deleted a test without admitting it.

## 🔴 The aliases removed in Python 3.12

`unittest` accumulated duplicate spellings over twenty years — `assertEquals`, `failUnless`,
`assertRegexpMatches`. They were deprecated in 3.2 and **removed in 3.12**.

If you are modernising old code (or reading a 2019 tutorial), this is the first thing that
breaks. The failure is an `AttributeError` at *call* time, so it only shows up when that test
runs.

In [ ]:
probe = unittest.TestCase()
removed_in_312 = [
    ("assertEqual",          "assertEquals"),
    ("assertNotEqual",       "assertNotEquals"),
    ("assertAlmostEqual",    "assertAlmostEquals"),
    ("assertTrue",           "failUnless"),
    ("assertRaises",         "failUnlessRaises"),
    ("assertRegex",          "assertRegexpMatches"),
    ("assertNotRegex",       "assertNotRegexpMatches"),
    ("assertItemsEqual  (py2 only, long gone)", "assertItemsEqual"),
]

print(f"  {'modern name':<42}{'old alias':<24}{'alias still exists'}")
print("  " + "-" * 84)
for modern, alias in removed_in_312:
    print(f"  {modern:<42}{alias:<24}{hasattr(probe, alias)}")

try:
    probe.assertEquals(1, 1)
except AttributeError as exc:
    print(f"\n  calling it: AttributeError: {exc}")

## Discovery — running the whole suite

You do not call `run_case` in real life. You point `unittest` at a directory and it finds the
tests itself.

```bash
python -m unittest discover                     # look in the current directory
python -m unittest discover -s tests -v         # look in tests/, verbosely
python -m unittest tests.test_jobs              # one module
python -m unittest tests.test_jobs.JobTests     # one class
python -m unittest tests.test_jobs.JobTests.test_new_job_starts_queued   # one test
```

Discovery rules, all overridable with `-p`:

- files matching **`test*.py`**
- inside them, classes subclassing **`TestCase`**
- inside those, methods starting with **`test`**

The next cell builds a small project in a temporary directory and runs discovery on it, so you
can see real output rather than a description of it.

In [ ]:
import subprocess
import textwrap

WORK = Path(tempfile.mkdtemp(prefix="py152proj_"))
(WORK / "jobs").mkdir()
(WORK / "tests").mkdir()

(WORK / "jobs" / "__init__.py").write_text("", encoding="utf-8")
(WORK / "jobs" / "queue.py").write_text(textwrap.dedent("""
    VALID = {"queued": {"running"}, "running": {"done", "failed"}, "done": set(),
             "failed": {"queued"}}

    def can_advance(current, new):
        return new in VALID.get(current, set())
"""), encoding="utf-8")

(WORK / "tests" / "__init__.py").write_text("", encoding="utf-8")
(WORK / "tests" / "test_queue.py").write_text(textwrap.dedent("""
    import unittest
    from jobs.queue import can_advance

    class TransitionTests(unittest.TestCase):
        def test_queued_can_start(self):
            self.assertTrue(can_advance("queued", "running"))

        def test_done_is_terminal(self):
            self.assertFalse(can_advance("done", "running"))

        def test_unknown_state_is_not_advanceable(self):
            self.assertFalse(can_advance("nonsense", "running"))
"""), encoding="utf-8")

(WORK / "tests" / "test_budget.py").write_text(textwrap.dedent("""
    import unittest

    class BudgetTests(unittest.TestCase):
        def test_placeholder(self):
            self.assertEqual(2 ** 3, 8)
"""), encoding="utf-8")

done = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
                      cwd=WORK, capture_output=True, text=True, timeout=120)
print(f"$ python -m unittest discover -s tests -v      (exit {done.returncode})\n")
print((done.stdout + done.stderr).rstrip())

🔴 **The most common discovery failure** is `ModuleNotFoundError: No module named 'jobs'`.
Discovery inserts the *start directory* on `sys.path`, not the project root — so running
`python -m unittest discover -s tests` from inside `tests/` cannot import your package.

Run it **from the project root** (as above), and keep an `__init__.py` in `tests/` or use
`--top-level-directory`. The same class of problem, with the same cause, appears in **15.3**
under the name *rootdir*, and properly in **17 Tooling, Packaging and Environments**.

## `unittest` or `pytest`?

| Reach for `unittest` when | Reach for `pytest` when |
|---|---|
| you cannot add a dependency | you can (almost always) |
| the codebase already uses it | you are starting fresh |
| you are writing a library and want zero test deps | you want readable failures |
| you need `mock` (which is `unittest.mock` — **15.5**) | you want fixtures and parametrisation |

Two things worth knowing before you choose:

1. **`pytest` runs `unittest` tests unchanged.** Adopting `pytest` does not mean rewriting
   anything — you get better output on the tests you already have, on day one.
2. **`unittest.mock` is the standard mocking library** regardless of which runner you use.
   `pytest` has no mocking of its own. That is **15.5**.

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
TRACE.clear()
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **`assertTrue(a == b)` instead of `assertEqual(a, b)`.** The first fails with “False is not true”; the second names both values. Always use the specific method.
2. 🔴 **`assertEqual` on floats.** `0.1 + 0.2 != 0.3`. Use `assertAlmostEqual` — see **2.2**.
3. **Using `assertEquals`, `failUnless` or `assertRegexpMatches`** — all removed in Python 3.12, and they fail only when that test runs.
4. **Calling `unittest.main()` in a notebook.** It calls `sys.exit()` and kills the kernel. Use `exit=False`, or load the case explicitly.
5. **Relying on test order.** Methods run alphabetically, not in definition order.
6. **Mutating `setUpClass` state.** It is created once and shared; one test's change is the next test's mystery.
7. **Wrapping several statements in one `assertRaises` block.** The test then passes if *any* of them raises — including the wrong one.
8. **A bare `for` loop over test cases.** The first failure hides the rest; use `subTest`.
9. **Skipping a test to silence it.** That is deleting a test while pretending otherwise. Use `@expectedFailure`, which tells you when the bug gets fixed.

## Best Practices

- Name the method after the claim: `test_failed_job_can_be_requeued`.
- Put cheap per-test setup in `setUp`; reserve `setUpClass` for genuinely expensive, genuinely read-only things.
- Prefer `addCleanup(...)` to `tearDown` — it runs even if `setUp` fails partway.
- Use `assertRaisesRegex` rather than `assertRaises` so a *different* error with the same type cannot pass the test.
- Reach for `subTest` for any table of cases, so you see every failure at once.
- Use `@expectedFailure` for known bugs — the suite then tells you when they are fixed.
- Run discovery from the project root, and keep tests in a `tests/` package.
- Remember `unittest.mock` is available to you whichever runner you use (**15.5**).

## Practice Exercises

Try these before moving on.

1. Write a `TestCase` for `Job.can_retry` covering: a failed job under budget, a failed job at budget, and a job that is not failed at all.
2. Convert the `CASES` table in this notebook to use `subTest`, then fix `parse_retry_after` so `'2.5'` and `' 5 '` both pass. What should `'abc'` do — return `0`, or raise? Defend your answer in the test name.
3. Add a `setUp` that creates a temporary spool directory and an `addCleanup` that removes it. Prove the isolation by writing a file in one test and asserting the directory is empty in another.
4. 🔴 Take `LifecycleTests` and move `self.job = Job(...)` from `setUp` into `setUpClass` as `cls.job`. Which test now fails, and why does it depend on alphabetical order?
5. Write a test using `assertLogs` that asserts the *exact* number of warnings logged when a job exhausts a budget of 3. Then use `assertNoLogs` to prove nothing is logged on the happy path.
6. Build the discovery project from this notebook on disk yourself and run `python -m unittest discover` from inside `tests/`. Reproduce the `ModuleNotFoundError`, then fix it two different ways.
7. **Interview question:** what is the difference between a *failure* and an *error*, and why might a framework choose not to distinguish them?

---

## Version notes

| Version | Change |
|---|---|
| **3.12** | 🔴 Deprecated aliases **removed**: `assertEquals`, `assertNotEquals`, `assertAlmostEquals`, `failUnless*`, `assertRegexpMatches`, `assertNotRegexpMatches` |
| **3.12** | `unittest.TestProgram` no longer accepts the removed `-p` shorthand spellings; use `--pattern` |
| **3.11** | `TestCase.enterContext()` added — enter a context manager for the test's lifetime |
| **3.11** | Deprecated aliases began emitting `DeprecationWarning` loudly before their 3.12 removal |
| **3.10** | `assertNoLogs()` added — assert that nothing was logged |
| **3.9** | `addClassCleanup()` added, the class-level partner of `addCleanup` |

## Where next

| Notebook | Covers |
|---|---|
| **15.3 pytest** | plain `assert` with real diffs, `@parametrize`, the CLI you will live in |
| **15.4 Fixtures** | `setUp` done properly — scopes, injection, `tmp_path` |
| **15.5 Test Doubles** | `unittest.mock`, which lives here but is used everywhere |

## Related

- **15.1** — `assert`, Arrange–Act–Assert, and the hand-rolled runner this replaces
- **05 OOPs** — classes, inheritance and `classmethod`, all used by `TestCase`
- **6.1 / 6.2** — the exceptions being asserted on here
- **2.2 Numeric Datatype** — why `assertAlmostEqual` exists
- **17 Tooling, Packaging and Environments** — `sys.path`, rootdir and import layout